In [51]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score


In [52]:
df=pd.read_csv("classification data.csv")

In [53]:
df.sample(20)

,DEPTH_MD,CALI,GR,PEF,DTC,SP,DRHO,LITHOLOGY
2173,1199.352000,13.813664,48.402481,4.686965,149.629669,49.155510,0.004941,sandstone
59178,1576.389599,13.174476,113.510849,4.393559,136.806839,87.807037,0.049857,shale
29971,977.677998,17.093800,40.761307,1.788790,142.346237,34.995453,0.016352,sandstone
59968,1703.157599,13.699133,125.566658,5.395841,143.638260,87.985497,0.047676,shale
10561,2474.936000,12.448548,8.608096,4.594848,62.654907,78.386917,-0.010450,sandstone
11671,2643.656000,10.886065,20.090221,4.258671,64.532082,102.400002,-0.031650,sandstone
36138,2014.925998,12.086001,37.567535,2.986202,140.137878,41.367100,-0.006832,sandstone
22860,2245.264000,12.811988,77.614761,2.593136,118.636650,82.511703,0.083650,sandstone
43388,1407.807790,14.783040,93.520576,33.628902,112.901108,84.673416,-0.604472,shale
7782,2051.920000,14.212404,47.156460,3.167955,142.375626,64.174591,-0.001829,sandstone


In [54]:
df.head()

,DEPTH_MD,CALI,GR,PEF,DTC,SP,DRHO,LITHOLOGY
0,869.056,18.544184,44.851158,1.576887,138.449585,29.880625,0.000899,sandstone
1,869.208,18.908665,37.774338,1.453022,140.845428,29.118149,-0.008508,sandstone
2,869.360,19.148600,33.735561,1.398047,142.352951,28.016756,-0.011523,sandstone
3,869.512,20.140171,30.677647,1.386652,142.852448,27.235287,-0.001811,sandstone
4,869.664,21.352367,29.963285,1.406170,143.337647,27.085091,0.017259,sandstone


In [55]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Step 1: Load dataset (Using a sample dataset for demonstration)
data =df.copy()

In [56]:

# Step 2: Encode the target variable
label_encoder = LabelEncoder()
data['LITHOLOGY'] = label_encoder.fit_transform(data['LITHOLOGY'])


In [57]:
df.head()

,DEPTH_MD,CALI,GR,PEF,DTC,SP,DRHO,LITHOLOGY
0,869.056,18.544184,44.851158,1.576887,138.449585,29.880625,0.000899,sandstone
1,869.208,18.908665,37.774338,1.453022,140.845428,29.118149,-0.008508,sandstone
2,869.360,19.148600,33.735561,1.398047,142.352951,28.016756,-0.011523,sandstone
3,869.512,20.140171,30.677647,1.386652,142.852448,27.235287,-0.001811,sandstone
4,869.664,21.352367,29.963285,1.406170,143.337647,27.085091,0.017259,sandstone


In [58]:
data.head()

,DEPTH_MD,CALI,GR,PEF,DTC,SP,DRHO,LITHOLOGY
0,869.056,18.544184,44.851158,1.576887,138.449585,29.880625,0.000899,0
1,869.208,18.908665,37.774338,1.453022,140.845428,29.118149,-0.008508,0
2,869.360,19.148600,33.735561,1.398047,142.352951,28.016756,-0.011523,0
3,869.512,20.140171,30.677647,1.386652,142.852448,27.235287,-0.001811,0
4,869.664,21.352367,29.963285,1.406170,143.337647,27.085091,0.017259,0


In [ ]:
# Step 3: Define Features and Target
X = data.iloc[:,:-1]
y = data['LITHOLOGY']

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import train_test_split, GridSearchCV

# Step 4: Split the dataset with a random state
random_state = 20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_state)

model = AdaBoostClassifier(random_state=42)

# Define grid of hyperparameters
param_grid = {
    'n_estimators': [50, 100],
    'learning_rate': [0.5, 1.0]
}

# Set up GridSearchCV
gcv = GridSearchCV(model, param_grid, cv=5, scoring='accuracy')

gcv.fit(X_train, y_train)


GridSearchCV(cv=5, estimator=DecisionTreeClassifier(random_state=42),
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [2, 3, 4, 5], 'min_samples_leaf': [1, 2],
                         'min_samples_split': [2, 3]},
             scoring='accuracy')

In [61]:
#1st way:
best_model=gcv.best_estimator_
y_pred = best_model.predict(X_test)

In [62]:
#2nd way:
y_pred = gcv.predict(X_test)

In [63]:
y_test

27008    0
12885    1
16525    0
54512    0
17816    1
        ..
23554    0
33475    0
28807    0
25755    0
35937    0
Name: LITHOLOGY, Length: 12498, dtype: int32

In [ ]:
save_folder = f"AB Classification_results_{random_state}"
os.makedirs(save_folder, exist_ok=True)

In [65]:
# Print best parameters
best_params = gcv.best_params_
print("Best Parameters:", best_params)
pd.DataFrame([best_params]).to_csv(os.path.join(save_folder, "best_params.csv"), index=False)

Best Parameters: {'criterion': 'gini', 'max_depth': 2, 'min_samples_leaf': 1, 'min_samples_split': 2}


In [66]:
# Print and save cross-validation results
cv_results = pd.DataFrame(gcv.cv_results_)
print("Cross-Validation Results:\n", cv_results)
cv_results.to_csv(os.path.join(save_folder, "cv_results.csv"), index=False)

Cross-Validation Results:
     mean_fit_time  std_fit_time  mean_score_time  std_score_time  \
0        0.057432      0.008344         0.000614        0.001228   
1        0.053688      0.007189         0.002731        0.004032   
2        0.049599      0.000406         0.003524        0.005788   
3        0.051543      0.002499         0.002266        0.004532   
4        0.050851      0.005041         0.006376        0.006824   
5        0.055398      0.011147         0.007406        0.007058   
6        0.059129      0.012614         0.000924        0.001848   
7        0.060884      0.015084         0.005326        0.005838   
8        0.057222      0.006418         0.000104        0.000208   
9        0.062680      0.014504         0.002650        0.002728   
10       0.065595      0.022248         0.004545        0.005880   
11       0.055470      0.007282         0.001546        0.001989   
12       0.056622      0.005214         0.003088        0.004916   
13       0.055451    

In [67]:

# Step 8: Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)


In [68]:
accuracy 

1.0

In [69]:
conf_matrix

array([[9187,    0],
       [   0, 3311]], dtype=int64)

In [70]:
precision 

1.0

In [71]:
recall

1.0

In [72]:
class_report 

'              precision    recall  f1-score   support\n\n           0       1.00      1.00      1.00      9187\n           1       1.00      1.00      1.00      3311\n\n    accuracy                           1.00     12498\n   macro avg       1.00      1.00      1.00     12498\nweighted avg       1.00      1.00      1.00     12498\n'

In [73]:




# Step 9: Save the metrics in a CSV file with folder name as random state
folder_name = str(random_state)
os.makedirs(folder_name, exist_ok=True)
result_file = os.path.join(folder_name, "model_metrics.csv")
pd.DataFrame({'Random_State': [random_state], 'Accuracy': [accuracy], 'Precision': [precision], 'Recall': [recall], 'F1_Score': [f1]}).to_csv(result_file, index=False)

# Step 10: Save confusion matrix separately
conf_matrix_file = os.path.join(folder_name, "confusion_matrix.csv")
pd.DataFrame(conf_matrix).to_csv(conf_matrix_file, index=False)

# Step 11: Save classification report
class_report_file = os.path.join(folder_name, "classification_report.csv")
with open(class_report_file, "w") as f:
    f.write(class_report)

print(f"Metrics saved in folder: {folder_name}, Files: model_metrics.csv, confusion_matrix.csv, classification_report.csv")

Metrics saved in folder: 20, Files: model_metrics.csv, confusion_matrix.csv, classification_report.csv


In [74]:
accuracy 

1.0

In [75]:
precision 

1.0

In [76]:
recall

1.0

In [77]:
f1

1.0

In [78]:
conf_matrix 

array([[9187,    0],
       [   0, 3311]], dtype=int64)

In [79]:
class_report 

'              precision    recall  f1-score   support\n\n           0       1.00      1.00      1.00      9187\n           1       1.00      1.00      1.00      3311\n\n    accuracy                           1.00     12498\n   macro avg       1.00      1.00      1.00     12498\nweighted avg       1.00      1.00      1.00     12498\n'